# Dify 电商客服评测 v4.3 — 全量评测（505条）

**用途**：P2路由修复 + P3检索优化 + 裁判修复后，全量505条评测。

**与回归测试版的区别**：
- Cell 2 直接加载完整 eval_set_v4.json（505条），不再提取失败子集
- 输出目录默认 eval_output_baseline_v3_full
- 评测完成后自动生成 baseline_v3_full_meta.json

**流程**：
1. Cell 1 配置（Dify URL + API Keys）
2. Cell 2 加载完整评测集（505条）
3. Cell 3 核心函数（Dify 调用 + 裁判打分 v4.3）
4. Cell 4 连通性测试（先跑 1 条）
5. Cell 5 批量评测（505条，约25分钟）
6. Cell 6 评测报告 + 标准化导出
7. Cell 7 保存结果文件
8. Cell 8 清理断点

## Cell 1: 配置

In [21]:
# ========== 配置 ==========
import os, sys, json, time, glob, requests
from getpass import getpass
from typing import Optional, List, Dict
from openai import OpenAI
import pandas as pd
from datetime import datetime
from collections import Counter

print("=== Dify 电商客服评测 v4.3 - 全量评测（505条） ===")

# Baseline 标识
BASELINE_ID = input("Baseline ID（默认 baseline_v3_full）: ").strip()
if not BASELINE_ID:
    BASELINE_ID = "baseline_v3_full"
BASELINE_NAME = input("Baseline 名称（默认 P2+P3修复后全量评测）: ").strip()
if not BASELINE_NAME:
    BASELINE_NAME = "P2+P3修复后全量评测"

# Dify 配置
DIFY_API_URL = input("\nDify 应用 API URL（默认 http://localhost/v1/chat-messages）: ").strip()
if not DIFY_API_URL:
    DIFY_API_URL = "http://localhost/v1/chat-messages"
DIFY_API_KEY = getpass("Dify 应用 API Key: ")

# 裁判模型配置
ALIYUN_API_KEY = getpass("阿里云百炼 API Key: ")
judge_client = OpenAI(
    api_key=ALIYUN_API_KEY,
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1"
)
JUDGE_MODEL = "qwen-plus"

# 完整评测集路径（505条那个）
EVAL_SET_PATH = input("\n完整评测集路径（默认 eval_set_v4.json）: ").strip()
if not EVAL_SET_PATH:
    EVAL_SET_PATH = "eval_set_v4.json"

# 输出目录
OUTPUT_DIR = input(f"结果输出目录（默认 eval_output_{BASELINE_ID}）: ").strip()
if not OUTPUT_DIR:
    OUTPUT_DIR = f"eval_output_{BASELINE_ID}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"\n✅ 配置完成")
print(f"   Baseline: {BASELINE_ID} ({BASELINE_NAME})")
print(f"   Dify URL: {DIFY_API_URL}")
print(f"   裁判模型: {JUDGE_MODEL}（阿里云百炼）")
print(f"   完整评测集: {EVAL_SET_PATH}")
print(f"   输出目录: {OUTPUT_DIR}")

=== Dify 电商客服评测 v4.3 - 全量评测（505条） ===


Baseline ID（默认 baseline_v3_full）:  
Baseline 名称（默认 P2+P3修复后全量评测）:  

Dify 应用 API URL（默认 http://localhost/v1/chat-messages）:  
Dify 应用 API Key:  ········
阿里云百炼 API Key:  ········

完整评测集路径（默认 eval_set_v4.json）:  
结果输出目录（默认 eval_output_baseline_v3_full）:  



✅ 配置完成
   Baseline: baseline_v3_full (P2+P3修复后全量评测)
   Dify URL: http://localhost/v1/chat-messages
   裁判模型: qwen-plus（阿里云百炼）
   完整评测集: eval_set_v4.json
   输出目录: eval_output_baseline_v3_full


## Cell 2: 加载完整评测集

直接加载 `eval_set_v4.json`（505条），不再提取失败子集。

In [19]:
# ========== 加载完整评测集（505条）==========
with open(EVAL_SET_PATH, 'r', encoding='utf-8') as f:
    eval_data = json.load(f)

eval_cases = eval_data['cases']
meta = eval_data.get('meta', {})

print(f"✅ 评测集加载完成")
print(f"   版本: {meta.get('version', '?')}")
print(f"   总用例数: {len(eval_cases)}")

type_dist = Counter(c['test_type'] for c in eval_cases)
cat_dist = Counter(c['category'] for c in eval_cases)
diff_dist = Counter(c['difficulty'] for c in eval_cases)

print(f"\n   按类型: {dict(type_dist)}")
print(f"   按分类: {dict(cat_dist)}")
print(f"   按难度: {dict(diff_dist)}")

# 预览前 5 条
print(f"\n📋 前 5 条用例预览:")
for c in eval_cases[:5]:
    print(f"   [{c.get('case_id','')}] {c.get('category','')} | {c.get('test_type','')} | {c.get('user_query','')[:40]}")

✅ 评测集加载完成
   版本: 4.0
   总用例数: 505

   按类型: {'single_turn': 382, 'multi_turn': 17, 'boundary': 35, 'prohibited': 23, 'adversarial': 48}
   按分类: {'商品咨询与规则查询': 279, '订单查询': 38, '退款换货': 71, '投诉': 47, '其他': 48, '边界混淆': 22}
   按难度: {'简单': 192, '中等': 182, '困难': 131}

📋 前 5 条用例预览:
   [TC-INTENT-001] 商品咨询与规则查询 | single_turn | 索尼WF-1000XM5蓝牙耳机的降噪效果怎么样？
   [TC-INTENT-002] 订单查询 | single_turn | 查订单 3178642857123452
   [TC-INTENT-003] 退款换货 | single_turn | 我要退款
   [TC-INTENT-004] 退款换货 | single_turn | 我要换货
   [TC-INTENT-005] 投诉 | single_turn | 我要投诉，你们服务太差了


## Cell 3: 核心函数定义

包含：
- `call_dify()` — 调用 Dify Chatflow API（单轮/多轮）
- `check_must_not()` — 确定性预检：禁止表述精确匹配
- `check_expected_points()` — 确定性预检：要点宽松匹配
- `llm_judge()` — 裁判模型打分（7维度 + 确定性预检覆盖）
- `evaluate_single_case()` — 单条用例完整评测流程
- `load_checkpoint()` / `save_checkpoint()` — 断点续评

In [20]:
# ========== 调用 Dify（单轮）==========
def call_dify(user_query: str, conversation_id: Optional[str] = None):
    headers = {
        "Authorization": f"Bearer {DIFY_API_KEY}",
        "Content-Type": "application/json"
    }
    data = {
        "inputs": {},
        "query": user_query,
        "user": "eval-user",
        "response_mode": "blocking"
    }
    if conversation_id:
        data["conversation_id"] = conversation_id
    try:
        resp = requests.post(DIFY_API_URL, headers=headers, json=data, timeout=60)
        resp.raise_for_status()
        result = resp.json()
        return result.get("answer", ""), result.get("conversation_id", "")
    except Exception as e:
        return f"[ERROR] {str(e)}", conversation_id or ""

# ========== 调用 Dify（多轮）==========
def call_dify_multi_turn(case: dict):
    context = case.get('conversation_context', [])
    if not context:
        return call_dify(case['user_query'])
    conversation_id = None
    for turn in context:
        if turn['role'] == 'user':
            _, conversation_id = call_dify(turn['content'], conversation_id)
            time.sleep(0.3)
    return call_dify(case['user_query'], conversation_id)

# ========== 确定性预检：must_not 精确子串匹配 ==========
def check_must_not(response: str, must_not_list: list) -> bool:
    """精确子串匹配，返回 True 表示违规"""
    if not must_not_list:
        return False
    for item in must_not_list:
        item = item.strip()
        if not item:
            continue
        if item in response:
            return True
    return False

# ========== 范围语义匹配辅助函数 ==========
def _is_range_match(point: str, response: str) -> bool:
    """范围语义匹配：如'1-7'匹配回复中出现的'1-3''3-7''2-5'等分段范围"""
    import re
    m = re.match(r'^(\d+)\s*[-~～]\s*(\d+)$', point.strip())
    if not m:
        return False
    p_start = int(m.group(1))
    p_end = int(m.group(2))
    
    resp_ranges = re.findall(r'(\d+)\s*[-~～]\s*(\d+)\s*(个工作日|天|小时|分钟)?', response)
    if not resp_ranges:
        return False
    
    all_starts = [int(r[0]) for r in resp_ranges]
    all_ends = [int(r[1]) for r in resp_ranges]
    
    # 回复中所有范围整体覆盖了期望范围 → 视为命中
    if all_starts and all_ends:
        min_start = min(all_starts)
        max_end = max(all_ends)
        if min_start <= p_start and max_end >= p_end:
            return True
    
    # 单个范围直接覆盖
    for rs, re_, unit in resp_ranges:
        if int(rs) <= p_start and int(re_) >= p_end:
            return True
    
    return False

# ========== 确定性预检：expected_points 宽松匹配 ==========
def check_expected_points(response: str, expected_points: list) -> bool:
    """宽松匹配：关键词或其变形出现在回复中即算命中，全部命中返回 True
    支持：精确子串、标点清理、范围语义匹配、前缀匹配、数字单位宽松匹配"""
    import re
    if not expected_points:
        return True
    for point in expected_points:
        point = point.strip()
        if not point:
            continue
        # 1. 精确匹配
        if point in response:
            continue
        # 2. 标点清理后匹配
        point_clean = point.replace("，", "").replace("。", "").replace("、", "").replace(" ", "")
        resp_clean = response.replace(" ", "")
        if point_clean in resp_clean:
            continue
        # 3. 范围语义匹配（如 "1-7" 匹配 "1-3/3-7" 等分段范围）
        if _is_range_match(point, response):
            continue
        # 4. 关键词前缀匹配（4字以上的词，取前4字）
        if len(point) >= 4:
            short = point[:4]
            if short in response:
                continue
        # 5. 数字+单位的宽松匹配（如 "7天" 匹配 "7个工作日"、"7日内"）
        num_match = re.match(r'^(\d+)(.+)$', point)
        if num_match:
            num = num_match.group(1)
            unit = num_match.group(2)
            pattern = num + r'[^0-9]{0,5}' + re.escape(unit[-1])
            if re.search(pattern, response):
                continue
        return False
    return True

# ========== 裁判评分（v4.3 修复版）==========
def llm_judge(case: dict, actual_response: str) -> dict:
    must_not_str = "、".join(case.get('must_not', [])) if case.get('must_not') else "无"
    expected_points = "、".join(case.get('expected_answer_contains', [])) if case.get('expected_answer_contains') else "无特殊要求"
    context_str = ""
    if case.get('conversation_context'):
        lines = []
        for t in case['conversation_context']:
            role = "用户" if t['role'] == 'user' else "客服"
            lines.append(f"{role}: {t['content'][:80]}")
        context_str = "\n".join(lines)
    complaint_level = case.get('expected_complaint_level') or "不适用"

    prompt = f"""你是一名严格的电商客服质量评审专家。请评估以下客服回复的质量。

## 评分规则（必须严格遵守）

### branch_routing（分支路由）评分规则
- 你无法看到系统的实际路由 trace，请根据回复的【行为模式】判断，而非回复的【话题内容】
- 判定方法：观察回复在"做什么"，而不是回复"在讲什么"
- 5分：回复的行为模式与期望分支"{case['expected_branch']}"完全一致
- 3-4分：行为模式基本符合期望分支，但有少量偏差
- 0-2分：行为模式明显属于其他分支

【各分支行为特征对照表】（核心判定依据）

1. 商品咨询与规则查询分支（知识检索 + LLM直接回答）：
   - 行为特征：直接解答用户的政策/规则/参数问题，不引导任何操作流程
   - 典型表现：直接给出答案（时效/运费/规则/参数等）、引用知识库内容、不索要订单号、不确认办理意向
   - 注意：即使回答的是"退款多久到账""退换货政策"等售后话题，只要是直接解答规则而非引导办理，就属于此分支
   - 操作流程咨询也属此分支：用户问"怎么操作/怎么申请/流程是什么"，回复描述操作步骤/方法 → 商品咨询（解答方法）
   - 知识库未命中也属此分支：回复"暂未收录/知识库没有相关信息" → 路由仍是商品咨询，只是信息完整性低

2. 退款换货分支（售后办理流程）：
   - 行为特征：引导用户完成退款/退货/换货/取消/修改的办理流程
   - 典型表现：确认退款/退货/换货意向、索要订单号、询问退款原因、引导寄回、告知办理步骤
   - 注意：如果只是回答退款政策问题（如时效），不引导办理，则不属于此分支
   - 告知办理结果也属此分支：如查询订单后告知"已超期无法办理""已签收无法取消" → 退款换货分支（已处理并给出结果）
   - 子场景包括：退款、退货、换货、取消订单、修改订单、价格保护申请等所有售后办理动作

3. 订单查询分支（工具调用查订单）：
   - 行为特征：查询并展示具体订单的实时数据
   - 典型表现：提供订单状态、物流信息、具体订单数据，通常需要订单号
   - 索要订单号以进行查询也属此分支
   - 注意：即使回复内容有幻觉（虚构物流信息），只要行为模式是在告知订单/物流状态，分支路由仍应判定为订单查询（幻觉由 faithfulness 维度负责）

4. 投诉分支（投诉处理流程）：
   - 行为特征：处理投诉/追责/赔偿诉求
   - 典型表现：道歉、记录投诉、升级处理、赔偿方案、安抚情绪
   - 投诉分支也可以转人工：作为升级处理的一种方式，不代表路由错误
   - 判断标准：是否有投诉处理的核心行为（道歉+记录/升级/补偿），转人工只是附加动作

5. 其他分支（兜底/转人工）：
   - 行为特征：无法归类、转人工、或通用兜底回复
   - 典型表现："抱歉我无法回答"、"为您转接人工"、身份说明
   - 注意：如果回复有明确的业务分支行为（如道歉+补偿=投诉、解答规则=商品咨询），即使附带转人工，也不应判为其他分支

【关键判定原则】
- 话题内容 ≠ 分支路由。退款话题可能走商品咨询分支（咨询政策），也可能走退款换货分支（办理退款）
- 核心区分：回复是在"提供信息/解答规则"还是"引导操作/办理流程"
- 直接解答政策/规则问题 → 商品咨询分支；引导办理/操作流程 → 对应业务分支
- 只要行为模式符合期望分支，即使回复内容涉及其他分支话题，也应给4分以上

【核心行为优先原则】（重要，避免误判）
- 当回复包含多种行为时，以【核心/主要行为】判定分支，不以附带/次要行为判定
- 判断方法：看回复的主体内容在做什么，开头和结尾的附加引导不算核心
- 示例 1：用户问"换货多久能收到"，回复先解答"3-5个工作日"（核心：规则解答），最后说"请提供订单号进一步核实"（附加：引导办理）→ 核心是商品咨询，branch_routing 4-5分
- 示例 2：用户问"质量问题能免费修吗"，回复先解答"质保期内非人为可免费修"（核心：规则解答），后半段引导"申请售后上传照片"（附加：引导办理）→ 核心是商品咨询，branch_routing 4-5分
- 示例 3：用户问"退货运费谁承担"，回复先解答"质量问题商家承担，无理由用户承担"（核心：规则解答），最后"请确认退货原因"（附加：轻微引导）→ 核心是商品咨询，branch_routing 4-5分

【操作流程咨询特殊判定】（重要，避免误判）
- 用户问"怎么操作/怎么申请/流程是什么/如何办理"等带疑问词的问题 → 属于操作流程咨询
- 回复描述了操作步骤/方法（如"点击申请售后→选择价格保护"）→ 商品咨询分支（解答方法）
- 回复引导用户现在就去办理（如"请提供订单号我帮您申请"）→ 退款换货分支（引导办理）
- 区分标准：回复是在"告诉用户怎么操作"（咨询）还是在"引导用户现在就办"（办理）
- 示例 1：用户问"怎么申请价格保护"，回复"在订单详情页点击申请售后→选择价格保护" → 商品咨询（解答方法）
- 示例 2：用户说"我要申请价格保护"，回复"请提供订单号" → 退款换货（引导办理）
- 示例 3：用户问"七天无理由退货怎么操作"，回复列出操作步骤 → 商品咨询（解答方法）

【大类与子场景区分原则】（重要，避免误判）
- 退款换货是一个大类，包含多个子场景：退款、退货、换货、修改订单、取消订单等
- branch_routing 只评判【大类】是否正确，不要求子场景完全匹配
- 只要回复的行为模式属于"售后办理引导"（确认需求、索要信息、引导流程、告知办理条件、告知办理结果等），即视为退款换货大类路由正确
- 子场景是否精准匹配（如改地址 vs 换货、取消订单 vs 退款），由 intent_accuracy 维度负责评分，branch_routing 不因此扣分
- 示例 1：用户说"改地址"，系统回复"已签收无法改地址，可考虑退货重拍" → branch_routing 4-5分（大类正确），intent_accuracy 可酌情扣分
- 示例 2：用户说"修改订单"，系统回复"请问要修改什么内容" → branch_routing 4-5分（大类正确），intent_accuracy 可酌情扣分
- 示例 3：用户说"我要退款"，系统回复"请问退款原因" → branch_routing 5分（大类和子场景都正确）

【超期/不符合条件告知的判定】（避免误判）
- 用户申请售后（退款/退货/换货），系统查询后告知"已超期/不符合条件/无法办理" → 属于退款换货分支的正确行为
- 系统已经处理了用户的诉求（查询订单+给出结论），只是结论是"无法办理"
- 不能因为系统没有"引导办理流程"就扣分，因为不符合条件时本就不应引导办理
- 示例：用户说"退订单 3178642857123451"，回复"已签收超过7天，无法取消，质量问题可申请质保" → branch_routing 4-5分（大类正确，已处理并给出结果）

【路由维度与其他维度的分离原则】（极其重要）
- branch_routing 只评判【行为模式属于哪个分支】，不评判内容的真实性、完整性、准确性
- 内容真实性 → faithfulness 维度负责
- 内容完整性 → information_completeness 维度负责
- 意图准确性 → intent_accuracy 维度负责
- 示例 1：订单查询分支回复了虚构的物流信息 → branch_routing 应高分（路由正确），faithfulness 应低分（幻觉）
- 示例 2：商品咨询分支回复"暂未收录" → branch_routing 应高分（路由正确），information_completeness 可低分（信息不足）
- 示例 3：投诉分支最后转了人工 → branch_routing 应高分（投诉行为是核心），不能因为转人工就判其他分支

### must_not_violation（禁止表述违规）判定规则
- 逐条检查 must_not 列表，使用精确子串匹配
- 只有当回复中完整包含 must_not 中的某个表述时，才判定为违规
- 禁止使用语义近似、同义词扩展来判断
- 否定表述不算违规：如 must_not=["可办理退货"]，回复中有"不可办理退货"→不违规（语义相反）
- 不同表述不算违规：如 must_not=["7天内可退"]，回复中有"7天无理由退货"→不违规（不同表述）

### contains_expected_points（要点覆盖）判定规则
- 逐条检查 expected_answer_contains 中的每个要点
- 使用宽松匹配：关键词的变形、包含该关键词的 longer phrase 均算命中
  如"未签收"匹配"尚未签收"、"运输中"匹配"目前运输中"
- 范围匹配：如期望"1-7天"，回复中有"1-3天"和"3-7天"等分段范围整体覆盖1-7区间，也视为命中
- 只要回复中体现了该要点的核心语义，即算命中
- 全部命中→true，有任何一个未命中→false

### boundary_correctness（边界正确性）评分规则
- 根据回复是否正确处理了边界值说明中的业务逻辑来评分
- 只要回复体现了边界值对应的核心处理逻辑，即给4分以上
- 不要因为回复提供了额外的替代方案就扣分

## 用例信息
【用例ID】{case['case_id']}
【测试类型】{case['test_type']}
【用户问题（最后一轮）】{case['user_query']}
【对话上下文】
{context_str if context_str else '无（单轮对话）'}

【期望意图】{case['expected_intent']}
【期望分支】{case['expected_branch']}
【期望IF/ELSE路径】{', '.join(case.get('expected_if_else_path', []))}
【期望答案应包含的关键要点】{expected_points}
【回复中禁止出现的表述】{must_not_str}
【期望工具调用】{', '.join(case.get('expected_tool_calls', [])) if case.get('expected_tool_calls') else '无'}
【期望投诉分级】{complaint_level}
【是否期望转人工】{case.get('expected_human_handoff', False)}
【边界值说明】{case.get('boundary_value') or '无'}

【实际客服回复】
{actual_response}

请从以下维度严格打分（0-5分），并输出 JSON 格式：
{{
    "intent_accuracy": <0-5>,
    "branch_routing": <0-5>,
    "answer_relevance": <0-5>,
    "information_completeness": <0-5>,
    "faithfulness": <0-5>,
    "tone_compliance": <0-5>,
    "must_not_violation": <true/false>,
    "boundary_correctness": <0-5>,
    "is_hallucination": <true/false>,
    "contains_expected_points": <true/false>,
    "handoff_correctness": <0-5, 不适用填-1>,
    "comment": "<80字以内>"
}}

只输出 JSON。"""

    # 第一优先：带 response_format 调用
    try:
        resp = judge_client.chat.completions.create(
            model=JUDGE_MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
            response_format={"type": "json_object"}
        )
        score = json.loads(resp.choices[0].message.content)
    except Exception as e1:
        # 第二优先：去掉 response_format 重试
        try:
            resp = judge_client.chat.completions.create(
                model=JUDGE_MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0
            )
            content = resp.choices[0].message.content.strip()
            if content.startswith("```"):
                content = content.split("\n", 1)[1] if "\n" in content else content[3:]
            if content.endswith("```"):
                content = content.rsplit("```", 1)[0]
            content = content.strip()
            if content.startswith("json"):
                content = content[4:].strip()
            score = json.loads(content)
        except Exception as e2:
            score = {
                "error": str(e1), "intent_accuracy": 0, "branch_routing": 0,
                "answer_relevance": 0, "information_completeness": 0,
                "faithfulness": 0, "tone_compliance": 0,
                "must_not_violation": False, "boundary_correctness": 0,
                "is_hallucination": False, "contains_expected_points": False,
                "handoff_correctness": -1, "comment": f"裁判调用失败: {e1}"
            }

    # ========== 确定性预检覆盖裁判误判 ==========
    must_not_list = case.get('must_not', [])
    expected_points_list = case.get('expected_answer_contains', [])

    code_must_not = check_must_not(actual_response, must_not_list)
    if not code_must_not and score.get('must_not_violation', False):
        score['must_not_violation'] = False
        orig_comment = score.get('comment', '')
        score['comment'] = f"[预检修正]must_not未违规(精确匹配)。{orig_comment}"

    code_points = check_expected_points(actual_response, expected_points_list)
    if code_points and not score.get('contains_expected_points', False):
        score['contains_expected_points'] = True
        orig_comment = score.get('comment', '')
        score['comment'] = f"[预检修正]要点已覆盖(宽松匹配)。{orig_comment}"

    return score

# ========== 单条执行 ==========
def evaluate_single_case(case: dict) -> dict:
    actual_response, conv_id = call_dify_multi_turn(case)
    score = llm_judge(case, actual_response)
    return {
        'case_id': case['case_id'],
        'test_type': case.get('test_type', ''),
        'category': case.get('category', ''),
        'difficulty': case.get('difficulty', ''),
        'user_query': case.get('user_query', ''),
        'expected_intent': case.get('expected_intent', ''),
        'expected_branch': case.get('expected_branch', ''),
        'expected_answer_contains': ','.join(case.get('expected_answer_contains', [])),
        'must_not': ','.join(case.get('must_not', [])),
        'expected_complaint_level': case.get('expected_complaint_level'),
        'expected_human_handoff': case.get('expected_human_handoff', False),
        'is_multi_turn': len(case.get('conversation_context', [])) > 0,
        'conversation_id': conv_id,
        'actual_response': actual_response,
        'score': score,
    }

# ========== 断点续评 ==========
def load_checkpoint():
    path = os.path.join(OUTPUT_DIR, "checkpoint.json")
    if os.path.exists(path):
        with open(path, 'r', encoding='utf-8') as f:
            return json.load(f)
    return []

def save_checkpoint(results):
    path = os.path.join(OUTPUT_DIR, "checkpoint.json")
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

print("✅ 核心函数定义完成（v4.3 - 行为特征路由判定 + 范围语义匹配）")

✅ 核心函数定义完成（v4.3 - 行为特征路由判定 + 范围语义匹配）


## Cell 4: 连通性测试

先跑 1 条用例，确认 Dify API 和裁判模型都通了再批量跑。

In [5]:
print("=== 连通性测试（第1条用例）===\n")
test_case = eval_cases[0]
result = evaluate_single_case(test_case)

print(f"用例ID: {test_case['case_id']}")
print(f"分类: {test_case.get('category','')}")
print(f"问题: {test_case['user_query']}")
print(f"\n回复: {result['actual_response'][:300]}")
print(f"\n评分:")
print(json.dumps(result['score'], ensure_ascii=False, indent=2))

# 判断连通性
score = result.get('score', {})
if 'error' in score:
    print(f"\n❌ 裁判模型调用失败: {score['error']}")
    print(f"   请检查 ALIYUN_API_KEY 是否正确")
elif result['actual_response'].startswith('[ERROR]'):
    print(f"\n❌ Dify API 调用失败: {result['actual_response']}")
    print(f"   请检查 DIFY_API_URL 和 DIFY_API_KEY")
else:
    print(f"\n✅ 连通性正常！继续下一个 Cell 批量评测")

=== 连通性测试（第1条用例）===

用例ID: TC-IF6-IF
分类: 投诉
问题: 快递延迟了两天还没到，我很着急

回复: 您好！非常抱歉给您带来了不便，快递延迟确实让您着急了。我们已安排加快配送进程，并为您准备了一张10元优惠券作为补偿，请注意查收。

请您耐心等待，我们会尽快处理。【投诉级别:L1| 处理方式：自动处理】

评分:
{
  "intent_accuracy": 5,
  "branch_routing": 5,
  "answer_relevance": 5,
  "information_completeness": 5,
  "faithfulness": 5,
  "tone_compliance": 5,
  "boundary_correctness": 5,
  "is_hallucination": false,
  "must_not_violation": false,
  "contains_expected_points": true,
  "comment": "[预检修正]must_not未违规。准确识别投诉意图，明确致歉与补偿（10元券），标注L1且无违规，边界（延迟两天）处理合理，无虚构信息。"
}

✅ 连通性正常！继续下一个 Cell 批量评测


## Cell 5: 批量评测

全量505条评测，约25分钟完成。
支持断点续评，每10条自动保存。

In [15]:
# ===== 可调参数 =====
START_INDEX = 0      # 从第几条开始（0-based）
BATCH_SIZE =  0       # 评测多少条（0=全部）
# ====================

results = load_checkpoint()
completed_ids = set(r['case_id'] for r in results)

if BATCH_SIZE > 0:
    eval_range = eval_cases[START_INDEX:START_INDEX + BATCH_SIZE]
else:
    eval_range = eval_cases[START_INDEX:]

pending = [c for c in eval_range if c['case_id'] not in completed_ids]
total = len(pending)

print(f"=== 批量评测（全量评测）===")
print(f"  总用例: {len(eval_cases)}（完整评测集）")
print(f"  已完成: {len(results)}（断点续评）")
print(f"  本次待评: {total}")
if total > 0:
    print(f"  预估耗时: ~{total * 3 // 60}分{total * 3 % 60}秒（505条约25分钟）")
print()

if total == 0:
    print("✅ 所有用例已完成")
else:
    start_time = time.time()
    for i, case in enumerate(pending, 1):
        elapsed = time.time() - start_time
        eta = (elapsed / (i-1) * (total - i + 1)) if i > 1 else total * 3
        print(f"[{i}/{total}] ({elapsed:.0f}s/{eta:.0f}s) {case['case_id']}...", end="", flush=True)
        try:
            r = evaluate_single_case(case)
            results.append(r)
            s = r['score']
            bs = s.get('branch_routing', 0)
            hal = s.get('is_hallucination', False)
            vio = s.get('must_not_violation', False)
            status = "✅" if bs >= 4 else "⚠️"
            flags = []
            if hal: flags.append("幻觉")
            if vio: flags.append("违规")
            fl = f"[{','.join(flags)}]" if flags else ""
            print(f" {status} 分支={bs}/5{fl}")
        except Exception as e:
            print(f" ❌ {e}")
            results.append({
                'case_id': case['case_id'],
                'category': case.get('category', ''),
                'user_query': case.get('user_query', ''),
                'expected_branch': case.get('expected_branch', ''),
                'actual_response': f"[ERROR] {e}",
                'score': {'branch_routing': 0, 'comment': str(e)}
            })
        if i % 10 == 0:
            save_checkpoint(results)
            print(f"  💾 已保存（{len(results)}条）")
        time.sleep(0.5)
    save_checkpoint(results)
    t = time.time() - start_time
    print(f"\n✅ 完成！共{len(results)}条，耗时{t//60:.0f}分{t%60:.0f}秒")

=== 批量评测（全量评测）===
  总用例: 505（完整评测集）
  已完成: 501（断点续评）
  本次待评: 4
  预估耗时: ~0分12秒（505条约25分钟）

[1/4] (0s/12s) TC-INTENT-008... ✅ 分支=5/5
[2/4] (16s/47s) TC-INTENT-010... ✅ 分支=5/5
[3/4] (40s/40s) TC-RULE-011... ✅ 分支=5/5
[4/4] (50s/17s) TC-BND-013... ✅ 分支=5/5

✅ 完成！共505条，耗时1分1秒


## Cell 6: 评测报告 ⭐ 核心输出

**这是整个脚本最重要的输出**：
- 通过率：505条用例的通过比例
- 失败用例：V3 仍然失败的用例 + 根因分析
- 7维度均分
- 按分类/类型/难度统计

In [16]:
# ========== 全量评测报告 ==========
total = len(results)

# 通过判定：branch_routing >= 4
pass_count = sum(1 for r in results if r.get('score', {}).get('branch_routing', 0) >= 4)
fail_count = total - pass_count
pass_rate = pass_count / total if total > 0 else 0

# 幻觉/违规统计
hal_count = sum(1 for r in results if r.get('score', {}).get('is_hallucination', False))
vio_count = sum(1 for r in results if r.get('score', {}).get('must_not_violation', False))

# 7维度均分
score_cols = ['intent_accuracy', 'branch_routing', 'answer_relevance',
              'information_completeness', 'faithfulness', 'tone_compliance', 'boundary_correctness']
avg_scores = {}
for col in score_cols:
    vals = [r.get('score', {}).get(col, 0) for r in results]
    avg_scores[col] = sum(vals) / len(vals) if vals else 0

# 四象限分析：本次通过/失败
results_ids = set(r['case_id'] for r in results)
passed_ids = set()      # 本次通过
not_passed_ids = set()  # 本次失败

for r in results:
    cid = r['case_id']
    bs = r.get('score', {}).get('branch_routing', 0)
    if bs >= 4:
        passed_ids.add(cid)
    else:
        not_passed_ids.add(cid)

pass_rate_100 = len(passed_ids) / total * 100 if total > 0 else 0

# ========== 打印报告 ==========
print(f"{'='*60}")
print(f"📊 全量评测报告 — {BASELINE_ID}")
print(f"{'='*60}")
print(f"\n  ╔══ 核心指标 ══╗")
print(f"  ║  评测总数: {total} 条")
print(f"  ║  本次通过: {pass_count} 条 ✅")
print(f"  ║  本次失败: {fail_count} 条 ❌")
print(f"  ║  通过率: {pass_rate*100:.1f}%")
print(f"  ║")
print(f"  ║  🎯 通过率: {pass_rate_100:.1f}%（{len(passed_ids)}/{total} 已通过）")
print(f"  ║  🔴 失败用例: {len(not_passed_ids)} 条（仍需修复）")
print(f"  ╚════════════════════╝")

print(f"\n  🚨 幻觉率: {hal_count/total*100:.1f}%")
print(f"  ⚠️  违规率: {vio_count/total*100:.1f}%")

print(f"\n--- 7维度均分 ---")
zh_names = {'intent_accuracy':'意图识别','branch_routing':'分支路由',
            'answer_relevance':'答案相关性','information_completeness':'信息完整性',
            'faithfulness':'忠实度','tone_compliance':'语气合规',
            'boundary_correctness':'边界正确性'}
for col in score_cols:
    print(f"  {zh_names.get(col,col):8s}: {avg_scores[col]:.2f}")

# 按分类统计修复情况
print(f"\n--- 按分类修复情况 ---")
cat_stats = {}
for r in results:
    cat = r.get('category', '未知')
    bs = r.get('score', {}).get('branch_routing', 0)
    if cat not in cat_stats:
        cat_stats[cat] = {'total':0, 'passed':0, 'failed_ids':[]}
    cat_stats[cat]['total'] += 1
    if bs >= 4:
        cat_stats[cat]['passed'] += 1
    else:
        cat_stats[cat]['failed_ids'].append(r['case_id'])

for cat, s in sorted(cat_stats.items()):
    rate = s['passed']/s['total']*100 if s['total']>0 else 0
    bar = '█' * int(rate/5) + '░' * (20-int(rate/5))
    print(f"  {cat:10s}: {bar} {s['passed']}/{s['total']} = {rate:.1f}%")

# ========== 失败用例完整详情 ==========
print(f"\n{'='*60}")
print(f"🔴 失败用例完整详情（共 {len(not_passed_ids)} 条）")
print(f"{'='*60}")

failed_list = [r for r in results if r['case_id'] in not_passed_ids]
# 按分支路由分从低到高排序，最严重的放前面
failed_list.sort(key=lambda r: r.get('score', {}).get('branch_routing', 0))

for i, r in enumerate(failed_list, 1):
    s = r.get('score', {})
    cid = r['case_id']
    cat = r.get('category', '')
    ttype = r.get('test_type', '')
    query = r.get('user_query', '')[:60]
    br = s.get('branch_routing', 0)
    ia = s.get('intent_accuracy', 0)
    ic = s.get('information_completeness', 0)
    fa = s.get('faithfulness', 0)
    bc = s.get('boundary_correctness', 0)
    hal = '🔴幻觉' if s.get('is_hallucination', False) else ''
    vio = '🔴违规' if s.get('must_not_violation', False) else ''
    pts = '✅要点全' if s.get('contains_expected_points', False) else '❌要点缺'
    comment = s.get('comment', '')[:80]
    resp_preview = r.get('actual_response', '')[:100].replace('\n', ' ')

    print(f"\n{'─'*58}")
    print(f"  [{i}/{len(not_passed_ids)}] {cid} | {cat} | {ttype}")
    print(f"  🗣️  用户问题: {query}")
    print(f"  📊  各维度: 意图={ia}/5  路由={br}/5  完整={ic}/5  忠实={fa}/5  边界={bc}/5")
    print(f"  🏷️  标签: {pts} {hal} {vio}")
    if comment:
        print(f"  💬  评语: {comment}")
    print(f"  📝  回复摘要: {resp_preview}...")

# ========== 根因分布 ==========
print(f"\n{'='*60}")
print(f"📋 失败根因分布")
print(f"{'='*60}")

root_causes = {
    '路由错误': [],
    '信息不完整': [],
    '忠实度/幻觉': [],
    '边界错误': [],
    '禁止表述违规': [],
    '其他': []
}

for r in failed_list:
    s = r.get('score', {})
    cid = r['case_id']
    br = s.get('branch_routing', 5)
    ic = s.get('information_completeness', 5)
    fa = s.get('faithfulness', 5)
    bc = s.get('boundary_correctness', 5)
    vio = s.get('must_not_violation', False)
    
    # 按优先级归类（路由错误最严重，优先归类）
    if br < 3:
        root_causes['路由错误'].append(cid)
    elif vio:
        root_causes['禁止表述违规'].append(cid)
    elif fa < 3 or s.get('is_hallucination', False):
        root_causes['忠实度/幻觉'].append(cid)
    elif bc < 3:
        root_causes['边界错误'].append(cid)
    elif ic < 4:
        root_causes['信息不完整'].append(cid)
    else:
        root_causes['其他'].append(cid)

for cause, ids in sorted(root_causes.items(), key=lambda x: -len(x[1])):
    if ids:
        print(f"\n  🔸 {cause}: {len(ids)} 条")
        print(f"     {ids}")

📊 全量评测报告 — baseline_v3_full

  ╔══ 核心指标 ══╗
  ║  评测总数: 505 条
  ║  本次通过: 490 条 ✅
  ║  本次失败: 15 条 ❌
  ║  通过率: 97.0%
  ║
  ║  🎯 通过率: 97.0%（490/505 已通过）
  ║  🔴 失败用例: 15 条（仍需修复）
  ╚════════════════════╝

  🚨 幻觉率: 2.6%
  ⚠️  违规率: 0.0%

--- 7维度均分 ---
  意图识别    : 4.77
  分支路由    : 4.88
  答案相关性   : 4.88
  信息完整性   : 4.54
  忠实度     : 4.89
  语气合规    : 4.98
  边界正确性   : 4.86

--- 按分类修复情况 ---
  其他        : ███████████████████░ 47/48 = 97.9%
  商品咨询与规则查询 : ███████████████████░ 271/279 = 97.1%
  投诉        : ████████████████████ 47/47 = 100.0%
  订单查询      : ██████████████████░░ 36/38 = 94.7%
  边界混淆      : ████████████████░░░░ 18/22 = 81.8%
  退款换货      : ████████████████████ 71/71 = 100.0%

🔴 失败用例完整详情（共 15 条）

──────────────────────────────────────────────────────────
  [1/15] TC-INTENT-015 | 订单查询 | single_turn
  🗣️  用户问题: 订单怎么还没到，太慢了
  📊  各维度: 意图=0/5  路由=0/5  完整=0/5  忠实=0/5  边界=5/5
  🏷️  标签: ❌要点缺 🔴幻觉 
  💬  评语: [预检修正]must_not未违规(精确匹配)。用户仅抱怨物流慢，未投诉或要求补发；客服擅自升级为投诉并承诺补发，属严重意图错判和事实幻觉。
  📝  回复摘要: 您好，非常抱歉您的订单还未送

## Cell 7: 保存结果文件

生成 4 个文件：
- `eval_results_{ts}.json` — 详细评分结果
- `eval_results_{ts}.csv` — CSV 格式（Excel 打开）
- `regression_report_{ts}.json` — 修复验证报告（结构化）
- `baseline_v3_meta.json` — 给 Module 5 对比用

In [17]:
ts = datetime.now().strftime("%Y%m%d_%H%M%S")

# 1. 详细结果 JSON
json_path = os.path.join(OUTPUT_DIR, f'eval_results_{BASELINE_ID}_{ts}.json')
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

# 2. CSV（Excel 友好）
rows = []
for r in results:
    s = r.get('score', {})
    rows.append({
        'case_id': r['case_id'],
        'category': r.get('category',''),
        'test_type': r.get('test_type',''),
        'difficulty': r.get('difficulty',''),
        'user_query': r.get('user_query','')[:80],
        'expected_branch': r.get('expected_branch',''),
        'actual_response': r.get('actual_response','')[:200],
        'intent_accuracy': s.get('intent_accuracy',0),
        'branch_routing': s.get('branch_routing',0),
        'answer_relevance': s.get('answer_relevance',0),
        'information_completeness': s.get('information_completeness',0),
        'faithfulness': s.get('faithfulness',0),
        'tone_compliance': s.get('tone_compliance',0),
        'boundary_correctness': s.get('boundary_correctness',0),
        'is_hallucination': s.get('is_hallucination',False),
        'must_not_violation': s.get('must_not_violation',False),
        'contains_expected_points': s.get('contains_expected_points',False),
        'comment': s.get('comment','')[:100],
    })
df = pd.DataFrame(rows)
csv_path = os.path.join(OUTPUT_DIR, f'eval_results_{BASELINE_ID}_{ts}.csv')
df.to_csv(csv_path, index=False, encoding='utf-8-sig')

# 3. baseline_v3_meta.json（给 Module 5 对比用）
meta = {
    "baseline_id": BASELINE_ID,
    "baseline_name": BASELINE_NAME,
    "created_at": datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    "judge_model": JUDGE_MODEL,
    "eval_set": EVAL_SET_PATH,
    "eval_scope": "full eval_set_v4 (505 cases)",
    "results": {
        "total_cases": total,
        "passed_cases": pass_count,
        "failed_cases": fail_count,
        "pass_rate": pass_rate,
        "passed_count": len(passed_ids),
        "failed_count": len(not_passed_ids),
    },
    "metrics": {
        "overall": {
            "route_accuracy": pass_rate,
            "intent_accuracy": avg_scores.get('intent_accuracy',0),
            "point_coverage": avg_scores.get('answer_relevance',0),
            "hallucination_rate": hal_count/total if total>0 else 0,
            "violation_rate": vio_count/total if total>0 else 0,
            "avg_intent_score": avg_scores.get('intent_accuracy',0),
            "avg_route_score": avg_scores.get('branch_routing',0),
            "avg_relevance_score": avg_scores.get('answer_relevance',0),
            "avg_completeness_score": avg_scores.get('information_completeness',0),
            "avg_faithfulness_score": avg_scores.get('faithfulness',0),
            "avg_tone_score": avg_scores.get('tone_compliance',0),
            "avg_boundary_score": avg_scores.get('boundary_correctness',0),
        }
    }
}
meta_path = os.path.join(OUTPUT_DIR, f'{BASELINE_ID}_meta.json')
with open(meta_path, 'w', encoding='utf-8') as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

# 4. 修复验证报告 JSON
report = {
    "meta": {
        "baseline": BASELINE_ID,
        "generated_at": datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        "total_regression": total,
        "pass_rate": pass_rate / 100,
        "passed_count": len(passed_ids),
        "failed_count": len(not_passed_ids),
    },
    "passed_ids": sorted(passed_ids),
    "not_passed_ids": sorted(not_passed_ids),
    "cat_stats": {k: {'total':v['total'],'passed':v['passed'],'failed_ids':v['failed_ids']}
                   for k,v in cat_stats.items()},
    "root_causes": {k:v for k,v in root_causes.items() if v},
}
report_path = os.path.join(OUTPUT_DIR, f'regression_report_{BASELINE_ID}_{ts}.json')
with open(report_path, 'w', encoding='utf-8') as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

# ========== 打印文件清单 ==========
print(f"\n{'='*60}")
print(f"💾 文件已保存:")
print(f"{'='*60}")
for fp in [json_path, csv_path, meta_path, report_path]:
    size = os.path.getsize(fp)
    print(f"   ✅ {os.path.basename(fp)}")
    print(f"      {fp} ({size:,} bytes)")

print(f"\n{'='*60}")
print(f"✅ V3 全量评测全部完成！")
print(f"{'='*60}")
print(f"\n📋 后续操作:")
print(f"   1. 将 {BASELINE_ID}_meta.json 放到 analysis_output/")
print(f"   2. 打开 Module 5 notebook")
print(f"   3. 对比 V1 → V2 → V3 三段式优化效果")
print(f"   4. 查看 V1 → V2 → V3 三段式优化效果对比")


💾 文件已保存:
   ✅ eval_results_baseline_v3_full_20260817_150803.json
      eval_output_baseline_v3_full\eval_results_baseline_v3_full_20260817_150803.json (710,504 bytes)
   ✅ eval_results_baseline_v3_full_20260817_150803.csv
      eval_output_baseline_v3_full\eval_results_baseline_v3_full_20260817_150803.csv (292,286 bytes)
   ✅ baseline_v3_full_meta.json
      eval_output_baseline_v3_full\baseline_v3_full_meta.json (1,024 bytes)
   ✅ regression_report_baseline_v3_full_20260817_150803.json
      eval_output_baseline_v3_full\regression_report_baseline_v3_full_20260817_150803.json (12,905 bytes)

✅ V3 全量评测全部完成！

📋 后续操作:
   1. 将 baseline_v3_full_meta.json 放到 analysis_output/
   2. 打开 Module 5 notebook
   3. 对比 V1 → V2 → V3 三段式优化效果
   4. 查看 V1 → V2 → V3 三段式优化效果对比


## Cell 8: 清理断点（需要重新评测时运行）

In [30]:
ckpt_path = os.path.join(OUTPUT_DIR, "checkpoint.json")
if os.path.exists(ckpt_path):
    os.remove(ckpt_path)
    print(f"✅ 断点已清理: {ckpt_path}")
    print(f"   现在可以重新运行 Cell 5 从头评测")
else:
    print(f"⚠️ 无断点文件")

✅ 断点已清理: eval_output_baseline_v3\checkpoint.json
   现在可以重新运行 Cell 5 从头评测


In [14]:
import json
import os

# checkpoint 文件路径（和 Cell 5 里 OUTPUT_DIR 一致）
checkpoint_path = "eval_output_baseline_v3_full/checkpoint.json"

# 要删除的 7 条失败用例 ID
ids_to_remove = [
     
     "TC-INTENT-008",
    "TC-RULE-011",
    "TC-INTENT-010",
    "TC-BND-013"
                    
]

# 读取
with open(checkpoint_path, "r", encoding="utf-8") as f:
    results = json.load(f)

print(f"删除前：{len(results)} 条")

# 过滤掉要删除的
before = len(results)
results = [r for r in results if r["case_id"] not in ids_to_remove]
after = len(results)

# 保存回去
with open(checkpoint_path, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"删除了：{before - after} 条")
print(f"删除后：{after} 条")
print(f"剩余 case_id 示例：{[r['case_id'] for r in results[:5]]}")

删除前：505 条
删除了：4 条
删除后：501 条
剩余 case_id 示例：['TC-INTENT-001', 'TC-INTENT-002', 'TC-INTENT-003', 'TC-INTENT-004', 'TC-INTENT-005']
